# Preparazione Dataset COCO — Pipeline Gemini → COCO

Questo notebook orchestra il pipeline completo:

```
Immagini PMCimages
    │
    ├─ Step 1: create_batch_requests()   → batch_requests.jsonl
    │
    ├─ Step 2: run_batch_inference()     → batch_results.jsonl  (richiede Gemini API key)
    │
    ├─ Step 3: jsonl_dir_to_coco()       → coco_all.json
    │
    └─ Step 4: split_coco()             → instances_train.json + instances_val.json
```

**Steps 1-2** richiedono `GEMINI_API_KEY` e connessione internet.  
**Steps 3-4** lavorano offline sui JSONL già scaricati.

I file COCO già generati sono disponibili in `data/` e possono essere usati direttamente.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config import (
    PMCIMAGES_DIR, RAW_BATCH_PREDICTIONS_DIR,
    COCO_ALL_PATH, COCO_TRAIN_PATH, COCO_VAL_PATH,
    DATA_DIR, GEMINI_MODEL, COCO_VAL_RATIO, COCO_SEED,
)
from src.batch_pipeline import (
    create_batch_requests,
    run_batch_inference,
    predictions_to_labelstudio,
)
from src.coco_converter import (
    jsonl_to_coco,
    jsonl_dir_to_coco,
    split_coco,
    save_coco,
)

# Directory di lavoro per i file temporanei di questo notebook
WORK_DIR = DATA_DIR / "batch_work"
WORK_DIR.mkdir(exist_ok=True)

print(f"Modello Gemini  : {GEMINI_MODEL}")
print(f"PMCimages dir   : {PMCIMAGES_DIR}")
print(f"Raw predictions : {RAW_BATCH_PREDICTIONS_DIR}")
print(f"COCO output     : {COCO_ALL_PATH}")

## Step 1 — Creazione richieste batch (richiede API key)

Salta questo step se le richieste sono già state create o se usi i JSONL già disponibili.

In [ ]:
import os
assert os.environ.get("GEMINI_API_KEY"), "Imposta la variabile GEMINI_API_KEY prima di procedere"

REQUESTS_JSONL = WORK_DIR / "batch_requests.jsonl"

create_batch_requests(
    images_dir   = PMCIMAGES_DIR,
    output_jsonl = REQUESTS_JSONL,
    model        = GEMINI_MODEL,
)
print("Step 1 commentato — usa i JSONL già disponibili in RAW_BATCH_PREDICTIONS_DIR")

## Step 2 — Esecuzione batch job Gemini (richiede API key)

Salta se i JSONL dei risultati sono già disponibili.

In [ ]:
RESULTS_JSONL = WORK_DIR / "batch_results.jsonl"

run_batch_inference(
    requests_jsonl = REQUESTS_JSONL,
    output_jsonl   = RESULTS_JSONL,
    model          = GEMINI_MODEL,
)
print("Step 2 commentato — usa i JSONL già disponibili")

## Step 3 — Conversione JSONL → COCO

Usa i JSONL già scaricati in `RAW_BATCH_PREDICTIONS_DIR`.

In [ ]:
# Verifica disponibilità dei file
jsonl_files = sorted(RAW_BATCH_PREDICTIONS_DIR.glob("*.jsonl"))
print(f"File JSONL disponibili: {len(jsonl_files)}")
for f in jsonl_files[:5]:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}  ({size_mb:.1f} MB)")
if len(jsonl_files) > 5:
    print(f"  ... e altri {len(jsonl_files) - 5}")

In [ ]:
# Converti tutti i JSONL in un unico dataset COCO
# ATTENZIONE: richiede alcuni minuti (32K immagini)

coco_all = jsonl_dir_to_coco(
    jsonl_dir  = RAW_BATCH_PREDICTIONS_DIR,
    images_dir = PMCIMAGES_DIR,
)

print(f"\nImmagini: {len(coco_all['images'])}")
print(f"Annotazioni: {len(coco_all['annotations'])}")

In [ ]:
# Salva coco_all.json (sovrascrive quello esistente solo se rigenerato)
# save_coco(coco_all, COCO_ALL_PATH)

# Alternativa: carica quello già esistente
import json
print("Carico il COCO già esistente...")
with open(COCO_ALL_PATH) as f:
    coco_all = json.load(f)
print(f"Immagini: {len(coco_all['images'])}")
print(f"Annotazioni: {len(coco_all['annotations'])}")

## Step 4 — Train/Val split

In [ ]:
train_coco, val_coco = split_coco(
    coco_all,
    val_ratio = COCO_VAL_RATIO,
    seed      = COCO_SEED,
)

In [ ]:
# Salva train e val
# Decommentare per sovrascrivere i file esistenti

# save_coco(train_coco, COCO_TRAIN_PATH)
# save_coco(val_coco,   COCO_VAL_PATH)

print(f"File già disponibili:")
print(f"  {COCO_TRAIN_PATH}")
print(f"  {COCO_VAL_PATH}")

## Bonus — Statistiche dataset

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

ann_per_image = Counter(a["image_id"] for a in coco_all["annotations"])
counts = list(ann_per_image.values())
# Aggiunge le immagini con 0 annotazioni
img_ids_with_ann = set(ann_per_image.keys())
img_ids_all = {img["id"] for img in coco_all["images"]}
zeros = len(img_ids_all - img_ids_with_ann)
counts += [0] * zeros

print(f"Immagini totali       : {len(coco_all['images'])}")
print(f"Annotazioni totali    : {len(coco_all['annotations'])}")
print(f"Media ann/immagine    : {np.mean(counts):.2f}")
print(f"Std dev               : {np.std(counts):.2f}")
print(f"Max ann in un'immagine: {max(counts)}")
print(f"Immagini senza ann    : {zeros}")

fig, ax = plt.subplots(figsize=(10, 4))
bins = range(0, min(max(counts) + 2, 50))
ax.hist(counts, bins=bins, color="#4A90D9", edgecolor="white")
ax.set_xlabel("Numero di annotazioni per immagine")
ax.set_ylabel("Numero di immagini")
ax.set_title("Distribuzione annotazioni per immagine (coco_all)")
ax.axvline(np.mean(counts), color="red", linestyle="--", label=f"media={np.mean(counts):.1f}")
ax.legend()
fig.tight_layout()
plt.show()